# AgentDrift · Part C — How we will evaluate

*EDA 3 / 3 · Evaluation protocol and synthesis of the data phase · ESILV 2026, group DIA 7*

---

### Where we are

Our guardrail must stop a hijacked AI agent before it executes the attacker's action. Part A described the data and its traps; part B checked the preparation that removes them. One question remains before training any model: **how do we measure whether the guardrail works, honestly?**

A detector is useful only if it works on **tasks it has never seen**. The evaluation therefore separates the data by **task template**:

| Part | Size | Role |
|---|---|---|
| **5 development folds** | 215 templates | train on four folds, measure on the fifth, five times: all model choices are made here |
| **Fixed test set** | 35 templates | kept aside, used once per final model, never to choose anything |
| **Unseen domain** | one domain at a time | train on four application domains, test on the fifth |

Two operating points are planned: the guardrail may raise a false alarm on **1%** of normal steps, or on only **0.2%**.

### What this notebook checks

1. Are the folds comparable? · 2. What can leak from training to evaluation? · 3. What does the unseen-domain test hold out? · 4. How many false alarms are left to measure? · 5. How precise will the test results be? · **6. Synthesis of the data phase** · Reproducibility check

> **Ground rules.** The test set only contributes counts (its size and attack rate). No score is ever computed on it: to anticipate the precision of the final results, we measure it on the development folds with a deliberately crude **reference score** (based only on the tool name, the step number and the domain — information the real models are not allowed to use). Every number quoted in the text is recomputed by the last cell.

In [1]:
import warnings

import pandas as pd

from agentdrift.data import audit as A
from agentdrift.pipelines.run_eda import INTERVAL_METRICS, compute_c
from agentdrift.utils.seed import set_seed
from agentdrift.viz import notebook as NB
from agentdrift.viz import plots as P

warnings.filterwarnings("ignore", category=FutureWarning)
set_seed()
P.setup()

dp = A.load_inputs_protocol()   # data with the fold of every trajectory and step
rc = compute_c(dp)              # all part C results (shared with `run_eda --part C`)

## 1. Are the folds comparable?

**Why it matters.** If one fold contained many more attacks than another, a model's score would change from fold to fold for reasons that have nothing to do with the model. The folds are built so that each one has the same mix of domains and attack types.

As a comparison, the orange dots show what happens if folds are built **by simulated world** instead of by task template.

In [2]:
folds = rc["folds"]
NB.show(P.plot_fold_prevalence(rc["fold_prevalence"]), "eda_c1_fold_prevalence")
NB.table(folds.rename(columns={"y_traj": "attacked trajectories", "y_scan": "steps after an injection",
                               "y_gate": "hijacked steps", "first_h": "first hijacked steps"}),
         pct_cols=["attacked trajectories", "steps after an injection", "hijacked steps"],
         int_cols=["templates", "trajectories", "worlds", "steps", "first hijacked steps"], caption="Size and attack rate per fold")

<Figure size 1210x308 with 3 Axes>

,templates,trajectories,worlds,steps,attacked trajectories,steps after an injection,hijacked steps,first hijacked steps
fold 0,43,"2,165",830,"12,193",43.3%,9.9%,16.6%,937
fold 1,44,"2,174",813,"12,267",44.6%,9.9%,17.9%,970
fold 2,42,"2,140",818,"12,121",44.6%,10.0%,17.3%,954
fold 3,45,"2,215",812,"12,565",43.2%,9.8%,17.3%,957
fold 4,41,"2,120",820,"12,082",44.5%,9.8%,17.4%,943
test,35,"1,722",756,"9,796",45.0%,10.1%,17.9%,775


In [3]:
NB.table(rc["balance"], dec_cols={"template folds": 1, "world folds": 1}, caption="Largest differences between folds (points)")

,template folds,world folds
max gap of a category share to the dev mean (pts),1.2,4.7
max gap of a domain share to the dev mean (pts),1.2,7.7
y_traj range across folds (pts),1.4,9.9
y_gate range across folds (pts),1.3,4.8
y_scan range across folds (pts),0.2,1.3


**What we see**
- Each fold holds 41 to 45 templates and 2,120 to 2,215 trajectories; the test set (35 templates, 1,722 trajectories) is a bit smaller than one fold.
- **Folds by task template are balanced**: the shares of domains and attack types stay within 1.2 points of the average, and the share of attacked trajectories varies by only 1.4 points.
- **Folds by world are not**: the share of attacked trajectories varies by 9.9 points and a domain's share by up to 7.7 points, because each world was generated around one attack type (part A).
- The test set's attack rates (17.9% hijacked steps, 10.1% steps after an injection) fall within the range of the folds.

**What it means.** A score that changes from one fold to another reflects the model and the tasks, not a different mix of attacks. Folds by world remain only a robustness check, to be read together with their own attack rate.

<sub>Traceability: `splits/grouped.py` · `configs/split.yaml`</sub>

## 2. What can leak from training to evaluation?

**Why it matters.** Anything shared between the training and the evaluation data can be memorised instead of learned. We check what the split shares and what it does not.

In [4]:
g = rc["grouping"]
labels = {
    "templates_in_two_folds_or_more": "task templates found in two folds or more",
    "templates_shared_dev_test": "task templates shared by development and test",
    "worlds_in_two_folds_or_more": "worlds found in two folds or more",
    "dev_worlds": "worlds in development",
    "test_trajectories_whose_world_is_in_dev": "test trajectories in a world seen in development",
    "trajectories_per_template_dev_median": "trajectories per template (median, development)",
}
pd.Series({labels[k]: g[k] for k in labels}, name="value").to_frame()

,value
task templates found in two folds or more,0.000
task templates shared by development and test,0.000
worlds found in two folds or more,1060.000
worlds in development,1136.000
test trajectories in a world seen in development,0.996
"trajectories per template (median, development)",51.000


**What we see**
- **No task template** is shared between two folds, nor between development and test: every evaluation is made on tasks the model has never seen.
- **Worlds are shared**: 1,060 of the 1,136 development worlds appear in several folds, and 99.6% of test trajectories take place in a world seen in development. A split cannot separate templates and worlds at the same time.
- A template groups about 51 trajectories. The real number of independent examples is therefore the number of **templates**, not of trajectories or steps (see §5).

**What it means.** The shared worlds are the only remaining leak, and the anonymisation removes it: part B showed that the text no longer reveals the world.

<sub>Traceability: leaks L1 (shared templates), L9 (shared worlds) · `tests/test_splits.py`</sub>

## 3. What does the unseen-domain test hold out?

**Why it matters.** A guardrail will be deployed on applications it was not trained on. We simulate this by training on four domains and testing on the fifth. But if the held-out domain has a different attack rate, part of the score change comes from that rate, not from the change of application.

In [5]:
lodo = rc["lodo"]
NB.table(lodo, pct_cols=[c for c in lodo.columns if c not in ("templates", "trajectories")],
         int_cols=["templates", "trajectories"], caption="Each domain held out, against the four training domains")

,templates,trajectories,share of dev,y_traj held out,y_traj train,y_gate held out,y_gate train,share delayed held out,share delayed train
banking,43,"1,819",16.8%,34.3%,46.0%,9.8%,18.8%,8.7%,6.9%
coding,43,"2,218",20.5%,45.6%,43.6%,18.5%,17.0%,6.8%,7.4%
email,43,"2,257",20.9%,46.3%,43.4%,19.4%,16.7%,6.9%,7.3%
medical,43,"2,224",20.6%,46.4%,43.4%,18.7%,16.9%,7.0%,7.3%
web,43,"2,296",21.2%,45.6%,43.6%,18.4%,17.0%,7.1%,7.3%


**What we see**
- Each held-out domain holds 43 templates and 17% to 21% of the development data.
- **Banking stands out**: 34% of its trajectories are attacked against 46% in the other domains, and 9.8% of its steps are hijacked against 18.8%. The other domains differ by less than 3 points.
- The share of delayed attacks, the hardest to catch, stays between 7% and 9% everywhere.

**What it means.** On held-out banking, a lower precision–recall score is expected from its lower attack rate alone. Results on an unseen domain are therefore always shown next to the domain's attack rate, and compared with measures that do not depend on it (ROC-AUC, actual false-alarm rate).

<sub>Traceability: hypotheses H1 (what transfers), H5 (false-alarm rate on a new domain)</sub>

## 4. How many false alarms are left to measure?

**Why it matters.** A guardrail that interrupts the user too often is switched off. We will set the alarm threshold so that only 1% (or 0.2%) of normal steps raise a false alarm. But a rate is only measurable if there are enough events: at 0.2% of a few thousand normal steps, only a handful of false alarms remain, and the measured rate becomes very uncertain.

The expected number of false alarms is simply *number of normal steps × target rate*. With *k* false alarms, the measured rate is uncertain by about 1/√*k* of its value.

In [6]:
res = rc["resolution"]
NB.show(P.plot_resolution(res, "0.2%", A.MIN_FALSE_POSITIVES), "eda_c2_resolution")
NB.table(res, int_cols=res.columns, caption="Positives, negatives and expected false alarms per evaluation unit")

<Figure size 880x708.4 with 1 Axes>

,templates,trajectories,negative trajectories,first H,gate +,gate −,scan +,scan −,gate FP @ 1%,scan FP @ 1%,gate FP @ 0.2%,scan FP @ 0.2%
fold 0,43,"2,165","1,228",937,"2,030","10,163","1,206","8,822",102,88,20,18
fold 1,44,"2,174","1,204",970,"2,196","10,071","1,216","8,877",101,89,20,18
fold 2,42,"2,140","1,186",954,"2,096","10,025","1,209","8,772",100,88,20,18
fold 3,45,"2,215","1,258",957,"2,175","10,390","1,226","9,124",104,91,21,18
fold 4,41,"2,120","1,177",943,"2,097","9,985","1,186","8,776",100,88,20,18
inner validation (median),58,"2,897","1,611","1,277","2,823","13,526","1,621","11,883",135,119,27,24
test,35,"1,722",947,775,"1,755","8,041",993,"7,081",80,71,16,14
held-out banking,43,"1,819","1,195",624,968,"8,941",872,"7,218",89,72,18,14
held-out coding,43,"2,218","1,206","1,012","2,334","10,257","1,272","9,101",103,91,21,18
held-out email,43,"2,257","1,213","1,044","2,513","10,461","1,308","9,409",105,94,21,19


**What we see**
- At **1%**, every evaluation unit keeps at least 71 false alarms to count: the measure is reliable everywhere.
- At **0.2%**, the test set leaves only about **16** false alarms for the gate and **14** for the scanner. The measured rate is then uncertain by about ±27% of its value. The threshold itself is set on an internal validation part where about 27 normal steps lie above it.
- The step the guardrail must stop — the first hijacked one — occurs 775 times in the test set.

**What it means (proposal).** 1% of normal steps is the main operating point; 0.2% is reported as a secondary point, always with the false-alarm rate actually observed, never as a guaranteed rate. A budget of false alarms **per trajectory** (947 normal test trajectories) is a more stable alternative.

<sub>Traceability: `docs/CONTEXT.md` §7.5 (thresholds) · decision 4 pending in `docs/PLAN_DONNEES.md`</sub>

## 5. How precise will the test results be?

**Why it matters.** The test set has only 35 templates. If a model scores 0.80 and another 0.82, is the difference real? Every result will be given with a **95% confidence interval**, obtained by resampling the test data many times (bootstrap). The question is *what* to resample: the steps of the same task tend to succeed or fail together, so resampling steps one by one would pretend we have far more independent examples than we do.

For each development fold, the reference score is resampled three ways: by **template** at the fold's size (the protocol), by **template** at the size of the test set (35 templates: the precision to expect on the test set), and by **step** (the wrong way, for comparison).

In [7]:
ci = rc["intervals"]
NB.show(P.plot_interval_widths({t: v["summary"] for t, v in ci.items()}, INTERVAL_METRICS), "eda_c3_interval_widths")
NB.table(ci["y_gate"]["summary"], dec_cols={c: 3 for c in ci["y_gate"]["summary"].columns},
         caption="Gate: estimate and width of the 95% interval (mean over the 5 folds)")

<Figure size 1210x418 with 2 Axes>

,estimate,sd across folds,"width (fold, templates)","width (test size, templates)","width (fold, steps iid)",design effect
metric,,,,,,
roc_auc,0.885,0.005,0.030,0.033,0.013,5.027
pr_auc,0.608,0.019,0.126,0.142,0.046,7.719
prevalence,0.173,0.004,0.029,0.032,0.013,4.697
recall@1%,0.165,0.020,0.127,0.137,0.031,16.309
fpr@1%,0.010,0.003,0.015,0.017,0.004,16.222
recall@0.2%,0.025,0.017,0.035,0.038,0.012,7.411
fpr@0.2%,0.001,0.002,0.003,0.004,0.001,5.517


Reading the metrics: **ROC-AUC** and **PR-AUC** measure how well the score ranks attacks above normal steps (PR-AUC focuses on the attacks); **recall@1%** is the share of hijacked steps caught when 1% of normal steps raise a false alarm.

**What we see** (reference score for the gate: PR-AUC 0.61, ROC-AUC 0.88)
- On a 35-template test set, a 95% interval spans about **±0.071 in PR-AUC** and **±0.068 in recall at 1%** for the gate (±0.050 and ±0.046 for the scanner). Two models closer than that can only be told apart by a **paired** comparison, made on the same resampled templates.
- Resampling steps instead of templates would make the intervals **2.8 times too narrow** in PR-AUC: the illusion of precision is large.
- Going from a fold (about 43 templates) to the test size (35) widens the intervals by only 9%.
- For ranking metrics (ROC-AUC, PR-AUC), the bootstrap is cautious: its uncertainty is 1.0 to 1.7 times the spread observed between folds.
- For metrics at a threshold, it is not always so: at 0.2%, the folds vary about **twice as much** as the bootstrap says (ratio 0.5 to 0.6). Each fold has its own threshold, while the bootstrap keeps one fixed: **the uncertainty of the threshold itself is not in the interval.**

These widths come from a weak score; a better model will have narrower intervals. What carries over is the gap between resampling templates and resampling steps.

In [8]:
NB.table(ci["y_gate"]["transfer"], pct_cols=ci["y_gate"]["transfer"].columns, pct_digits=2,
         caption="Gate: false-alarm rate on the training folds and on the next fold, same threshold")

**What we see.** A threshold set for 1% of false alarms gives between 0.17% and 1.97% on the next fold (gate and scanner together), and a 0.2% threshold gives between 0.00% and 0.59%. Part of it comes from the reference score, which takes few distinct values (on the training folds it already reaches only 0.47% to 0.98% for the 1% target); the rest is the change of tasks.

**What it means.**
- Every test result is given with an interval obtained by **resampling templates** (2,000 resamples), and results at a threshold also with their spread over the five folds, which includes the uncertainty of the threshold.
- Models are chosen on the average over the five folds, never on the test set; two models are compared on the same resampled templates.
- The spread of the false-alarm rate between folds is the **reference** for the unseen-domain test: only a larger gap can be attributed to the change of domain.

<sub>Traceability: `evaluation/bootstrap.py` · `docs/CONTEXT.md` §8.3 · hypothesis H5</sub>

## 6. Synthesis of the data phase

The three parts of the EDA lead to a set of decisions, taken before training any model. For each: what we found, what we decided, and where it stands.

| What we found | Evidence | What we decided | Status |
|---|---|---|---|
| Tasks shared between training and test would inflate scores | 0 template shared after the split | Split by task template; test set of 35 unseen templates | ✅ implemented |
| The simulated world gives the answer away | world lookup alone: F1 0.838 | Anonymise names, companies and addresses, keeping relations | ✅ implemented |
| Worlds cannot be separated together with templates | 1,060 of 1,136 worlds in several folds | Check that the text no longer reveals the world | ✅ implemented |
| The anonymised text no longer reveals the world | text probe 12.2% → 3.1% (chance 2.1%) | Models read the anonymised data only | ✅ implemented |
| The step number alone predicts the attacks | ROC-AUC 0.876 for the scanner | Never a model input; baseline to beat | 🔄 automatic test to add |
| The total length leaks the future | ROC-AUC 0.58 | Models only see the past of each step | ✅ implemented |
| Each attack type has its own wording tics | tics in 99% of full injections, 8% after masking | Mask headers, policy codes and amounts with general rules | ✅ implemented |
| Even without tics, the text tells the attack type | probe 95.2% after masking (chance 25%) | Compare models with and without masking; external test on ATBench | 📋 decided |
| Partial and delayed injections are often unreadable | readable: 100% full, 65% partial, 46% delayed | Annotate 300 injections by hand; report recall by readability | 🔄 in progress |
| The agent's reasoning confesses the attack | 75% of hijacked steps of partial attacks | Train without the reasoning by default | ✅ implemented |
| Attacks are moderately rare; what matters is the first hijacked step | 17.4% of steps hijacked; 5,536 first hijacked steps | Precision–recall metrics; count the first alarm | 📋 decided |
| The scanner can only warn earlier on delayed attacks | 93% of delayed attacks act 2 steps later | Measure the warning delay on delayed attacks only | 📋 decided |
| The test set is small | 35 templates; ±0.07 in PR-AUC | Intervals by resampling templates; choices on the 5 folds | ✅ implemented |
| Steps of a task are not independent | intervals 2.8× too narrow if steps are resampled | Always resample templates | ✅ implemented |
| 0.2% of false alarms is hardly measurable | about 16 false alarms on the test set | 1% as main point, 0.2% as secondary | 💬 proposed |
| Domains differ in attack rate | banking: 34% attacked against 46% | Unseen-domain test read with the domain's attack rate | 📋 decided |

✅ implemented · 🔄 in progress · 📋 decided, implemented with the evaluation code · 💬 proposed, to confirm as a team

**The data is ready for the first models**: they will read anonymised and masked trajectories, step by step, without the step number nor anything that comes after, and be evaluated on unseen tasks with honest confidence intervals. What remains — the two hand annotations — completes the preparation without blocking the first baselines.

## Reproducibility check

This cell recomputes every number quoted in the text and stops the notebook if one of them no longer matches the data.

In [9]:
import json

from agentdrift.config import RUNS
from agentdrift.viz.notebook import num, pct

dev, b = folds.drop(index=A.UNIT_TEST), rc["balance"]
low = lodo["y_traj held out"].idxmin()
gs, ss = ci["y_gate"]["summary"], ci["y_scan"]["summary"]
both = pd.concat({"y_gate": gs, "y_scan": ss})
ratio = (both["width (fold, templates)"] / 3.92) / both["sd across folds"]
metric = ratio.index.get_level_values(1)
tr = pd.concat({t: v["transfer"] for t, v in ci.items()})
held, train = tr.xs("held-out fold", axis=1, level=1), tr.xs("training folds", axis=1, level=1)
ma = json.loads((RUNS / "eda" / "metrics.json").read_text())
mb = json.loads((RUNS / "eda_b" / "metrics.json").read_text())
cell = lambda t, r, c: t["data"][t["index"].index(r)][t["columns"].index(c)]
w = lambda s, m: f"±{s.loc[m, 'width (test size, templates)'] / 2:.3f}"
NB.check([
    ("§1 templates / trajectories per fold", f"{dev['templates'].min()} to {dev['templates'].max()} / {num(dev['trajectories'].min())} to {num(dev['trajectories'].max())}",
     "41 to 45 / 2,120 to 2,215"),
    ("§1 test templates / trajectories", f"{folds.loc['test', 'templates']} / {num(folds.loc['test', 'trajectories'])}", "35 / 1,722"),
    ("§1 template folds: type / domain / attacked (points)",
     " / ".join(f"{b.loc[k, 'template folds']:.1f}" for k in ("max gap of a category share to the dev mean (pts)",
                                                              "max gap of a domain share to the dev mean (pts)", "y_traj range across folds (pts)")),
     "1.2 / 1.2 / 1.4"),
    ("§1 world folds: attacked / domain (points)",
     f"{b.loc['y_traj range across folds (pts)', 'world folds']:.1f} / {b.loc['max gap of a domain share to the dev mean (pts)', 'world folds']:.1f}", "9.9 / 7.7"),
    ("§1 test hijacked / after injection", f"{pct(folds.loc['test', 'y_gate'], 1)} / {pct(folds.loc['test', 'y_scan'], 1)}", "17.9% / 10.1%"),
    ("§2 templates shared (folds / dev-test)", f"{g['templates_in_two_folds_or_more']} / {g['templates_shared_dev_test']}", "0 / 0"),
    ("§2 worlds in several folds", f"{num(g['worlds_in_two_folds_or_more'])} of {num(g['dev_worlds'])}", "1,060 of 1,136"),
    ("§2 test in a known world", pct(g["test_trajectories_whose_world_is_in_dev"], 1), "99.6%"),
    ("§2 trajectories per template", f"{g['trajectories_per_template_dev_median']:.0f}", "51"),
    ("§3 templates / share of dev", f"{lodo['templates'].min():.0f} / {pct(lodo['share of dev'].min())} to {pct(lodo['share of dev'].max())}", "43 / 17% to 21%"),
    ("§3 lowest domain", low, "banking"),
    ("§3 banking attacked vs others", f"{pct(lodo.loc[low, 'y_traj held out'])} / {pct(lodo.loc[low, 'y_traj train'])}", "34% / 46%"),
    ("§3 banking hijacked steps vs others", f"{pct(lodo.loc[low, 'y_gate held out'], 1)} / {pct(lodo.loc[low, 'y_gate train'], 1)}", "9.8% / 18.8%"),
    ("§3 other domains differ by less than",
     f"{100 * (lodo.drop(index=low)['y_traj held out'] - lodo.drop(index=low)['y_traj train']).abs().max():.0f}", "3"),
    ("§3 delayed share", f"{pct(lodo['share delayed held out'].min())} to {pct(lodo['share delayed held out'].max())}", "7% to 9%"),
    ("§4 fewest false alarms at 1%", f"{res['scan FP @ 1%'].min():.0f}", "71"),
    ("§4 test false alarms at 0.2% gate / scanner", f"{res.loc['test', 'gate FP @ 0.2%']:.0f} / {res.loc['test', 'scan FP @ 0.2%']:.0f}", "16 / 14"),
    ("§4 uncertainty (scanner, test)", f"±{100 / res.loc['test', 'scan FP @ 0.2%'] ** 0.5:.0f}%", "±27%"),
    ("§4 internal validation at 0.2%", f"{res.loc['inner validation (median)', 'gate FP @ 0.2%']:.0f}", "27"),
    ("§4 first hijacked steps in test", f"{res.loc['test', 'first H']:.0f}", "775"),
    ("§4 normal test trajectories", f"{res.loc['test', 'negative trajectories']:.0f}", "947"),
    ("§5 reference score PR-AUC / ROC-AUC", f"{gs.loc['pr_auc', 'estimate']:.2f} / {gs.loc['roc_auc', 'estimate']:.2f}", "0.61 / 0.88"),
    ("§5 gate PR-AUC / recall@1%", f"{w(gs, 'pr_auc')} / {w(gs, 'recall@1%')}", "±0.071 / ±0.068"),
    ("§5 scanner PR-AUC / recall@1%", f"{w(ss, 'pr_auc')} / {w(ss, 'recall@1%')}", "±0.050 / ±0.046"),
    ("§5 step resampling too narrow", f"{gs.loc['pr_auc', 'width (fold, templates)'] / gs.loc['pr_auc', 'width (fold, steps iid)']:.1f}", "2.8"),
    ("§5 fold to test size", pct((gs.loc[INTERVAL_METRICS, "width (test size, templates)"] / gs.loc[INTERVAL_METRICS, "width (fold, templates)"] - 1).mean()), "9%"),
    ("§5 templates per fold (median)", f"{dev['templates'].median():.0f}", "43"),
    ("§5 ranking metrics ratio", f"{ratio[metric.isin(['roc_auc', 'pr_auc'])].min():.1f} to {ratio[metric.isin(['roc_auc', 'pr_auc'])].max():.1f}", "1.0 to 1.7"),
    ("§5 ratio at 0.2%", f"{ratio[metric.str.endswith('@0.2%')].min():.1f} to {ratio[metric.str.endswith('@0.2%')].max():.1f}", "0.5 to 0.6"),
    ("§5 next fold at 1% / 0.2%", f"{pct(held['1%'].min(), 2)} and {pct(held['1%'].max(), 2)} / {pct(held['0.2%'].min(), 2)} and {pct(held['0.2%'].max(), 2)}",
     "0.17% and 1.97% / 0.00% and 0.59%"),
    ("§5 training folds at 1%", f"{pct(train['1%'].min(), 2)} to {pct(train['1%'].max(), 2)}", "0.47% to 0.98%"),
    ("§5 resamples", num(rc["bootstrap"]["n_resamples"]), "2,000"),
    ("§6 world lookup F1", f"{cell(ma['shortcuts'], 'world lookup → y_traj', 'f1'):.3f}", "0.838"),
    ("§6 text probe", f"{pct(cell(mb['world_probe'], 'raw', 'world_accuracy'), 1)} → {pct(cell(mb['world_probe'], 'anonymised', 'world_accuracy'), 1)} "
     f"({pct(cell(mb['world_probe'], 'raw', 'chance_given_category'), 1)})", "12.2% → 3.1% (2.1%)"),
    ("§6 position only (scanner)", f"{cell(ma['shortcuts'], 'position only → y_scan', 'roc_auc'):.3f}", "0.876"),
    ("§6 length only", f"{cell(ma['shortcuts'], 'trajectory length → y_traj', 'roc_auc'):.2f}", "0.58"),
    ("§6 readable full / partial / delayed",
     " / ".join(pct(cell(ma["visibility"], c, "any of the three")) for c in ("attacked_full", "attacked_partial", "attacked_delayed")), "100% / 65% / 46%"),
    ("§6 hijacked steps / first hijacked",
     f"{pct(cell(ma['prevalence_split'], 'all', 'y_gate'), 1)} / {num(cell(ma['prevalence_split'], 'all', 'first_h'))}", "17.4% / 5,536"),
    ("§6 confession words, partial attacks",
     pct(next(r for r in ma["reproduction"]["data"] if r[ma["reproduction"]["columns"].index("measure")] == "confession words in thought, H partial")
         [ma["reproduction"]["columns"].index("recomputed")]), "75%"),
    ("§6 tics after masking (full)", pct(cell(mb["injection_signals"], "full", ["generator tic (opener or policy code)", "masked"])), "8%"),
    ("§6 attack-type probe after masking", pct(cell(mb["attack_type_probe"], "masked", "accuracy"), 1), "95.2%"),
    ("§6 delayed acting 2 steps later",
     pct(ma["gap"]["data"][ma["gap"]["index"].index("attacked_delayed")][ma["gap"]["columns"].index(2)]
         / sum(ma["gap"]["data"][ma["gap"]["index"].index("attacked_delayed")])), "93%"),
])

number quoted in the text,recomputed,quoted,ok
§1 templates / trajectories per fold,"41 to 45 / 2,120 to 2,215","41 to 45 / 2,120 to 2,215",✓
§1 test templates / trajectories,"35 / 1,722","35 / 1,722",✓
§1 template folds: type / domain / attacked (points),1.2 / 1.2 / 1.4,1.2 / 1.2 / 1.4,✓
§1 world folds: attacked / domain (points),9.9 / 7.7,9.9 / 7.7,✓
§1 test hijacked / after injection,17.9% / 10.1%,17.9% / 10.1%,✓
§2 templates shared (folds / dev-test),0 / 0,0 / 0,✓
§2 worlds in several folds,"1,060 of 1,136","1,060 of 1,136",✓
§2 test in a known world,99.6%,99.6%,✓
§2 trajectories per template,51,51,✓
§3 templates / share of dev,43 / 17% to 21%,43 / 17% to 21%,✓


All 42 numbers quoted in the text match the data.
